In [ ]:
# 1. Install Library Utama
!pip install ultralytics roboflow

import torch
import os
import shutil
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import files

print("--- VALIDASI HARDWARE ---")
if torch.cuda.is_available():
    print(f"✅ GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU TIDAK TERDETEKSI! Jangan lanjut, ubah Runtime ke T4 GPU terlebih dahulu.")

In [ ]:
# Masukkan API Key dan detail project Soybean v2 kamu

rf = Roboflow(api_key="T6bbNG6UtainlEBiRdRI")
project = rf.workspace("gpts-workspace-c64zv").project("kulit-kedelai")
version = project.version(2)
dataset = version.download("yolov11")


# Mengambil path data.yaml secara otomatis
path_config = os.path.join(dataset.location, "data.yaml")
print(f"✅ Dataset Soybean siap! File konfigurasi ada di: {path_config}")

In [ ]:
# 1. Load Pretrained Model untuk Tuning
tune_model = YOLO('yolo11m-seg.pt')

print("--- MEMULAI HYPERPARAMETER TUNING (RESOLUSI FULL 1560) ---")
# Proses GA Mutation sebanyak 50 iterasi
tune_results = tune_model.tune(
    data=path_config,
    epochs=20,              # 20 epoch per iterasi (standar tuning)
    iterations=50,          # 50 kombinasi hyperparameter berbeda
    optimizer="AdamW",
    imgsz=1560,             # MENGIKUTI RESOLUSI ASLI DATASET KEDELAI KAMU
    rect=True,              # Mempertahankan aspek rasio asli 1560x980 (efisiensi piksel)
    plots=True,
    save=True,
    val=True,
    device=0,               # Pakai GPU Colab
    batch=2,                # Menggunakan batch 2 agar aman dari crash/OOM di resolusi 1560
    workers=4,
    project="runs/segment",
    name="tune_soybean_1560",
    exist_ok=True
)
print("--- TUNING RESOLUSI 1560 SELESAI ---")

In [ ]:
folder_tune = 'runs/segment/tune_soybean_1560'
zip_tune = 'hasil_tuning_soybean_1560.zip'

if os.path.exists(folder_tune):
    shutil.make_archive('hasil_tuning_soybean_1560', 'zip', folder_tune)
    print("✅ Hasil tuning resolusi 1560 berhasil di-zip. Mendownload ke PC...")
    files.download(zip_tune)
else:
    print("❌ Folder hasil tuning tidak ditemukan.")

In [ ]:
# 1. Load Ulang Model Pretrained Bersih
final_model = YOLO('yolo11m-seg.pt')

print("--- MEMULAI TRAINING FINAL SOYBEAN (100 EPOCHS - RESOLUSI 1560) ---")
final_model.train(
    data=path_config,
    epochs=100,              # Training penuh untuk kematangan akurasi
    imgsz=1560,              # Resolusi penuh 1560px
    rect=True,               # Tetap mempertahankan aspek rasio 1560x980
    batch=2,                 # Jika Colab kamu crash/OOM, turunkan ke 1
    device=0,
    workers=4,
    amp=True,

    # --- MASUKKAN HASIL PARAMETER TERBAIK DARI PROSES TUNING KAMU ---
    lr0=0.0009,
    lrf=0.01,
    momentum=0.95832,
    weight_decay=0.00045,
    warmup_epochs=2.22486,
    warmup_momentum=0.78434,
    box=7.30174,
    cls=0.52127,
    dfl=2.5662,
    hsv_h=0.02657,
    hsv_s=0.74738,
    hsv_v=0.49611,
    degrees=0.01132,
    translate=0.10077,
    scale=0.39973,
    shear=0.00003,
    perspective=0.00021,
    flipud=0.01435,
    fliplr=0.53519,
    mosaic=0.85061,
    mixup=0.00444,
    copy_paste=0.00369,
    close_mosaic=7,          # Menonaktifkan mosaic pada 7 epoch terakhir agar segmentasi rapi

    project="runs/segment",
    name="soybean_final_1560",
    exist_ok=True
)
print("--- FINAL TRAINING SELESAI ---")

In [ ]:
from IPython.display import Image, display

chart_path = 'runs/segment/soybean_final_1560/results.png'
if os.path.exists(chart_path):
    print("Grafik Performa Model Kulit Kedelai (1560px):")
    display(Image(filename=chart_path))
else:
    print("Grafik belum digenerate. Tunggu training berjalan hingga beberapa epoch.")

In [ ]:
folder_final = 'runs/segment/soybean_final_1560'
zip_final = 'soybean_model_final_1560px.zip'

print("--- PROSES EXPORT MODEL FINAL SOYBEAN ---")
if os.path.exists(folder_final):
    # Proses Kompresi Folder
    shutil.make_archive('soybean_model_final_1560px', 'zip', folder_final)
    print("Mendownload file zip ke PC... Harap tunggu hingga pop-up download browser muncul.")
    files.download(zip_final)
else:
    print("⚠️ Folder hasil training final `soybean_final_1560` tidak ditemukan!")